# BillionMetaLab ASS 字幕特效量化分析

## Phase 1：数据提取 → SQLite 数据库

运行以下 Cell 遍历所有 `.ass` 文件，提取基础结构指标与特效标签统计，存入 `ass_stats.db`。

### Cell 1: Imports & 路径配置

In [ ]:
import sqlite3, re
from pathlib import Path
from collections import Counter

ROOT_DIR = Path('..').resolve()
DB_PATH = Path('ass_stats.db').resolve()

print(f'ROOT_DIR: {ROOT_DIR}')
print(f'DB_PATH:  {DB_PATH}')

### Cell 2: Module 1 — File Scanning

In [ ]:
def scan_ass_files(root_dir):
    """递归扫描所有 .ass 文件，排除 .git 目录。"""
    files = []
    for f in Path(root_dir).rglob('*.ass'):
        if '.git' not in f.parts:
            files.append(f)
    return sorted(files)


def classify_lang_type(filename):
    """从文件名判断语言类型。简日/日简统一为简日，繁日/日繁统一为繁日。"""
    f = filename.lower()
    if any(p in f for p in ['chs&jpn', 'chs&jp', '简日', 'chs_jpn', 'jp&chs', 'jpn&chs']):
        return 'chs_jpn'
    if any(p in f for p in ['cht&jpn', 'cht&jp', '繁日', 'cht_jpn', 'jp&cht', 'jpn&cht']):
        return 'cht_jpn'
    if any(p in f for p in ['[chs]', '.chs.', 'chs.', '[chs&', '简体']):
        return 'chs'
    if any(p in f for p in ['[cht]', '.cht.', 'cht.', '[cht&', '繁体']):
        return 'cht'
    if any(p in f for p in ['.jpn.', '.jp.', '[jpn]', '[jp]']):
        return 'jpn'
    return 'unknown'


def parse_file_path(file_path, root_dir):
    """从文件路径提取 season, anime_name, lang_type 等元数据。"""
    rel = file_path.relative_to(Path(root_dir))
    parts = rel.parts
    season = parts[0] if parts else 'unknown'
    if len(parts) >= 3:
        anime_name, file_name = parts[1], '/'.join(parts[2:])
    elif len(parts) == 2:
        anime_name, file_name = parts[0], parts[1]
    else:
        anime_name, file_name = 'root', (parts[0] if parts else '')
    lang_type = classify_lang_type(str(file_path))
    return {
        'file_path': str(rel), 'season': season, 'anime_name': anime_name,
        'file_name': file_name, 'lang_type': lang_type,
        'is_bilingual': 1 if '_' in lang_type else 0,
    }

### Cell 3: Module 2 — ASS File Parsing

In [ ]:
def parse_ass_file(file_path):
    """解析 ASS 文件，返回 (sections_dict, lines_list)。"""
    content = None
    for enc in ['utf-8-sig', 'utf-8', 'gbk', 'shift-jis']:
        try:
            with open(file_path, 'r', encoding=enc) as fh:
                content = fh.read()
            break
        except (UnicodeDecodeError, UnicodeError):
            continue
    if content is None:
        raise ValueError(f'Cannot decode: {file_path}')
    lines = content.split('\n')
    sections, cur_sec, cur_lines = {}, None, []
    for line in lines:
        s = line.strip()
        if s.startswith('[') and s.endswith(']'):
            if cur_sec is not None:
                sections[cur_sec] = '\n'.join(cur_lines)
            cur_sec, cur_lines = s[1:-1], []
        else:
            cur_lines.append(line)
    if cur_sec is not None:
        sections[cur_sec] = '\n'.join(cur_lines)
    return sections, lines


def split_ass_csv(line):
    """分割 ASS CSV 行，尊重 {...} 嵌套。"""
    result, current, depth = [], [], 0
    for ch in line:
        if ch == ',' and depth == 0:
            result.append(''.join(current).strip())
            current = []
        else:
            if ch == '{': depth += 1
            elif ch == '}': depth -= 1
            current.append(ch)
    result.append(''.join(current).strip())
    return result


def parse_styles(styles_text):
    """解析 [V4+ Styles] → [{...}, ...]。"""
    styles, fmt_cols = [], []
    if not styles_text:
        return styles
    for line in styles_text.split('\n'):
        line = line.strip()
        if line.startswith('Format:'):
            fmt_cols = [c.strip() for c in line[len('Format:'):].split(',')]
        elif line.startswith('Style:'):
            vals = split_ass_csv(line[len('Style:'):])
            if len(vals) >= len(fmt_cols):
                styles.append(dict(zip(fmt_cols, vals[:len(fmt_cols)])))
    return styles


def parse_events(events_text):
    """解析 [Events] → [{..., 'type':'Dialogue'|'Comment'}, ...]。"""
    events, fmt_cols = [], []
    if not events_text:
        return events
    for line in events_text.split('\n'):
        line = line.strip()
        if line.startswith('Format:'):
            fmt_cols = [c.strip() for c in line[len('Format:'):].split(',')]
        elif line.startswith('Dialogue:') or line.startswith('Comment:'):
            etype = 'Dialogue' if line.startswith('Dialogue:') else 'Comment'
            vals = split_ass_csv(line[len(etype)+1:])
            event = dict(zip(fmt_cols, vals[:len(fmt_cols)]))
            event['type'] = etype
            events.append(event)
    return events


def extract_override_tags(text):
    """提取文本中所有 {...} 覆盖块。"""
    return re.findall(r'\{([^}]*)\}', text)

### Cell 4: Module 3 — Effect Tag Classification

In [ ]:
# 标签名 → 类别 映射表（单一 dict 替代冗长 if-elif）
TAG_CATEGORY = {
    'p':'draw','p1':'draw','p2':'draw','p3':'draw','p4':'draw','pbo':'draw',
    'k':'karaoke','kf':'karaoke','ko':'karaoke','K':'karaoke',
    't':'transform','move':'move','clip':'clip','iclip':'clip',
    'fad':'fade','fade':'fade',
    'c':'color','1c':'color','2c':'color','3c':'color','4c':'color',
    'alpha':'alpha','1a':'alpha','2a':'alpha','3a':'alpha','4a':'alpha',
    'bord':'border','shad':'border','xbord':'border','ybord':'border','xshad':'border','yshad':'border',
    'pos':'position','an':'position','org':'position',
    'fs':'font_size','fscx':'font_size','fscy':'font_size',
    'frz':'rotate','frx':'rotate','fry':'rotate',
    'blur':'blur','fsp':'spacing','fn':'font',
}

CATEGORY_WEIGHT = {'draw':3.0,'transform':2.0,'karaoke':2.0,'move':1.0,'clip':1.0}


def get_tag_base(seg):
    """提取标签基础名。e.g. '3c&HFFFFFF&'→'3c', 'bord1.6'→'bord'"""
    seg = seg.strip()
    if re.match(r'\d[ca]', seg):
        return seg[:2]
    m = re.match(r'([a-zA-Z]+)', seg)
    return m.group(1) if m else None


def classify_tags_in_block(tag_block):
    """分类单个 {...} 块中所有标签 → Counter({category: count})。"""
    categories = Counter()
    for seg in tag_block.split('\\'):
        base = get_tag_base(seg.strip())
        if base and base in TAG_CATEGORY:
            categories[TAG_CATEGORY[base]] += 1
    return categories


def compute_effect_stats(dialogues):
    """对 Dialogue 列表做全量特效统计。"""
    stats = {
        'total_effect_tags':0,'dialogues_with_fx':0,'unique_tag_types':set(),
        'tag_draw':0,'tag_karaoke':0,'tag_transform':0,
        'tag_move':0,'tag_clip':0,'tag_fade':0,
        'tag_color':0,'tag_alpha':0,'tag_border':0,
        'tag_position':0,'tag_font_size':0,'tag_rotate':0,
        'tag_blur':0,'tag_spacing':0,'tag_font':0,
    }
    for d in dialogues:
        if d.get('type') != 'Dialogue':
            continue
        blocks = extract_override_tags(d.get('Text',''))
        if blocks:
            stats['dialogues_with_fx'] += 1
        for block in blocks:
            stats['total_effect_tags'] += 1
            for cat, cnt in classify_tags_in_block(block).items():
                key = f'tag_{cat}'
                if key in stats:
                    stats[key] += cnt
                    stats['unique_tag_types'].add(cat)
    stats['unique_tag_types'] = len(stats['unique_tag_types'])
    return stats


def compute_complexity_score(stats, dialogue_count=None):
    """计算 0–10 综合复杂度评分。"""
    score = 0.0
    for cat, w in CATEGORY_WEIGHT.items():
        if stats.get(f'tag_{cat}',0) > 0:
            score += w
    dc = dialogue_count or 1
    if stats.get('total_effect_tags',0) / max(dc,1) > 3.0:
        score += 1.0
    return min(score, 10.0)

### Cell 5: Module 4 — Database Operations

In [ ]:
SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS file_stats (
    id INTEGER PRIMARY KEY AUTOINCREMENT, file_path TEXT NOT NULL,
    season TEXT NOT NULL, anime_name TEXT NOT NULL, file_name TEXT NOT NULL,
    lang_type TEXT NOT NULL DEFAULT 'unknown', total_lines INTEGER DEFAULT 0,
    style_count INTEGER DEFAULT 0, dialogue_count INTEGER DEFAULT 0,
    comment_count INTEGER DEFAULT 0, is_bilingual INTEGER DEFAULT 0);
CREATE TABLE IF NOT EXISTS effect_stats (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    file_id INTEGER NOT NULL REFERENCES file_stats(id),
    total_effect_tags INTEGER DEFAULT 0, dialogues_with_fx INTEGER DEFAULT 0,
    unique_tag_types INTEGER DEFAULT 0,
    tag_draw INTEGER DEFAULT 0, tag_karaoke INTEGER DEFAULT 0, tag_transform INTEGER DEFAULT 0,
    tag_move INTEGER DEFAULT 0, tag_clip INTEGER DEFAULT 0, tag_fade INTEGER DEFAULT 0,
    tag_color INTEGER DEFAULT 0, tag_alpha INTEGER DEFAULT 0, tag_border INTEGER DEFAULT 0,
    tag_position INTEGER DEFAULT 0, tag_font_size INTEGER DEFAULT 0, tag_rotate INTEGER DEFAULT 0,
    tag_blur INTEGER DEFAULT 0, tag_spacing INTEGER DEFAULT 0, tag_font INTEGER DEFAULT 0,
    complexity_score REAL DEFAULT 0.0);
CREATE TABLE IF NOT EXISTS season_summary (
    season TEXT PRIMARY KEY, file_count INTEGER DEFAULT 0, anime_count INTEGER DEFAULT 0,
    total_lines INTEGER DEFAULT 0, avg_styles REAL DEFAULT 0.0,
    avg_fx_per_dialogue REAL DEFAULT 0.0, fx_dialogue_ratio REAL DEFAULT 0.0,
    files_with_draw INTEGER DEFAULT 0, files_with_karaoke INTEGER DEFAULT 0,
    avg_complexity REAL DEFAULT 0.0);
CREATE TABLE IF NOT EXISTS effect_distribution (
    tag_category TEXT PRIMARY KEY, files_using INTEGER DEFAULT 0,
    usage_ratio REAL DEFAULT 0.0, total_occurrences INTEGER DEFAULT 0);
CREATE INDEX IF NOT EXISTS idx_eff_fid ON effect_stats(file_id);
CREATE INDEX IF NOT EXISTS idx_f_season ON file_stats(season);
CREATE INDEX IF NOT EXISTS idx_f_lang ON file_stats(lang_type);
"""

ALL_CATEGORIES = ['draw','karaoke','transform','move','clip','fade',
                  'color','alpha','border','position','font_size','rotate','blur','spacing','font']


def init_database(db_path):
    conn = sqlite3.connect(str(db_path))
    conn.executescript(SCHEMA_SQL); conn.commit()
    return conn


def _insert(conn, table, data):
    """通用 INSERT 辅助函数。"""
    keys = ','.join(data.keys())
    placeholders = ','.join([':'+k for k in data])
    conn.execute(f'INSERT INTO {table} ({keys}) VALUES ({placeholders})', data)
    return conn.lastrowid


def aggregate_season_summary(cur):
    cur.execute('DELETE FROM season_summary')
    cur.execute("""
        INSERT INTO season_summary SELECT f.season, COUNT(*), COUNT(DISTINCT f.anime_name),
        SUM(f.total_lines), ROUND(AVG(f.style_count),2),
        ROUND(AVG(CAST(e.total_effect_tags AS REAL)/MAX(f.dialogue_count,1)),4),
        ROUND(AVG(CAST(e.dialogues_with_fx AS REAL)/MAX(f.dialogue_count,1)),4),
        SUM(CASE WHEN e.tag_draw>0 THEN 1 ELSE 0 END),
        SUM(CASE WHEN e.tag_karaoke>0 THEN 1 ELSE 0 END),
        ROUND(AVG(e.complexity_score),2)
        FROM file_stats f JOIN effect_stats e ON f.id=e.file_id GROUP BY f.season
    """)


def aggregate_effect_distribution(cur):
    cur.execute('DELETE FROM effect_distribution')
    total = cur.execute('SELECT COUNT(*) FROM file_stats').fetchone()[0]
    for cat in ALL_CATEGORIES:
        col = f'tag_{cat}'
        row = cur.execute(f"SELECT COUNT(*),COALESCE(SUM({col}),0) FROM effect_stats WHERE {col}>0").fetchone()
        cur.execute('INSERT OR REPLACE INTO effect_distribution VALUES(?,?,?,?)',
                    (cat, row[0], round(row[0]/max(total,1),4), row[1]))

### Cell 6: Module 5 — 主提取流程

In [ ]:
def run_extraction(root_dir, db_path):
    """完整提取流水线。返回处理文件数。"""
    root, dbp = Path(root_dir), Path(db_path)
    if dbp.exists():
        dbp.unlink()
    conn = init_database(dbp)
    cur = conn.cursor()
    files = scan_ass_files(root)
    total = len(files)
    print(f'Found {total} .ass files\n')
    errors = []
    for i, fp in enumerate(files):
        try:
            meta = parse_file_path(fp, root)
            sections, all_lines = parse_ass_file(fp)
            meta['total_lines'] = len(all_lines)
            meta['style_count'] = len(parse_styles(sections.get('V4+ Styles','')))
            events = parse_events(sections.get('Events',''))
            dialogues = [e for e in events if e.get('type')=='Dialogue']
            comments  = [e for e in events if e.get('type')=='Comment']
            meta['dialogue_count'] = len(dialogues)
            meta['comment_count'] = len(comments)
            fx_stats = compute_effect_stats(events)
            fx_stats['complexity_score'] = compute_complexity_score(fx_stats, len(dialogues))
            fid = _insert(cur, 'file_stats', meta)
            fx_stats['file_id'] = fid
            _insert(cur, 'effect_stats', fx_stats)
        except Exception as exc:
            errors.append((fp.name, str(exc)))
        if (i+1)%50==0 or (i+1)==total:
            print(f'  [{i+1:>4}/{total}] processed ...')
    print('\nAggregating ...')
    aggregate_season_summary(cur)
    aggregate_effect_distribution(cur)
    conn.commit()
    n = cur.execute('SELECT COUNT(*) FROM file_stats').fetchone()[0]
    t = cur.execute('SELECT SUM(total_lines) FROM file_stats').fetchone()[0]
    s = cur.execute('SELECT COUNT(DISTINCT season) FROM file_stats').fetchone()[0]
    print(f'\nDone. Files={n}, Lines={t:,}, Seasons={s}, Errors={len(errors)}')
    if errors:
        for fn, err in errors[:5]:
            print(f'  ERR: {fn}: {err}')
    conn.close()
    return n

### Cell 7: 执行提取

In [ ]:
count = run_extraction(ROOT_DIR, DB_PATH)
print(f'\nDone — {count} files written to {DB_PATH}')

---

## Phase 2：数据分析 & 量化展示

从 `ass_stats.db` 读取数据，生成 6 张量化汇总表格。

### Cell 8: 加载数据库

In [ ]:
import sqlite3
from pathlib import Path
from IPython.display import display, Markdown

DB_PATH = Path('ass_stats.db').resolve()
conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row

n_files  = conn.execute('SELECT COUNT(*) FROM file_stats').fetchone()[0]
n_lines  = conn.execute('SELECT SUM(total_lines) FROM file_stats').fetchone()[0]
n_season = conn.execute('SELECT COUNT(DISTINCT season) FROM file_stats').fetchone()[0]
n_anime  = conn.execute('SELECT COUNT(DISTINCT anime_name) FROM file_stats').fetchone()[0]
avg_cx   = conn.execute('SELECT ROUND(AVG(complexity_score),2) FROM effect_stats').fetchone()[0]
zero_fx  = conn.execute('SELECT COUNT(*) FROM effect_stats WHERE complexity_score=0').fetchone()[0]

print(f'DB: {DB_PATH}')
print(f'Files: {n_files} | Lines: {n_lines:,} | Seasons: {n_season} | Anime: {n_anime}')
print(f'Avg complexity: {avg_cx}/10 | Zero-FX: {zero_fx} ({zero_fx/n_files*100:.1f}%)')

### Table 1: 季度维度汇总

In [ ]:
rows = conn.execute('''
    SELECT season, file_count, anime_count, total_lines,
           avg_styles, avg_fx_per_dialogue, fx_dialogue_ratio,
           files_with_draw, files_with_karaoke, avg_complexity
    FROM season_summary ORDER BY season''').fetchall()

display(Markdown('### Table 1: 季度维度汇总'))
lines = ['| 季度 | 文件数 | 番剧数 | 总行数 | 平均Style | 行均特效标签 | 特效行占比 | 含绘图 | 含卡拉OK | 平均复杂度 |',
         '|------|--------|--------|--------|-----------|-------------|-----------|--------|----------|------------|']
for r in rows:
    lines.append(f'| {r["season"]} | {r["file_count"]} | {r["anime_count"]} | {r["total_lines"]:,} | '
                 f'{r["avg_styles"]} | {r["avg_fx_per_dialogue"]} | {r["fx_dialogue_ratio"]:.1%} | '
                 f'{r["files_with_draw"]} | {r["files_with_karaoke"]} | {r["avg_complexity"]} |')
display(Markdown('\n'.join(lines)))

### Table 2: 特效类型全局分布

In [ ]:
rows = conn.execute('''SELECT tag_category, files_using, usage_ratio, total_occurrences
                       FROM effect_distribution ORDER BY total_occurrences DESC''').fetchall()

CAT_CN = {
    'draw':'矢量绘图 \\p','karaoke':'卡拉OK \\k','transform':'变换动画 \\t',
    'move':'移动 \\move','clip':'裁剪遮罩 \\clip','fade':'淡入淡出 \\fad',
    'color':'颜色覆盖 \\c','alpha':'透明度 \\alpha','border':'边框阴影 \\bord',
    'position':'定位对齐 \\pos','font_size':'字体大小 \\fs',
    'rotate':'旋转 \\fr','blur':'模糊 \\blur','spacing':'间距 \\fsp','font':'字体 \\fn'
}

display(Markdown('### Table 2: 特效类型全局分布'))
lines = ['| 特效类别 | 使用文件数 | 使用占比 | 总出现次数 |',
         '|----------|-----------|---------|------------|']
for r in rows:
    label = CAT_CN.get(r['tag_category'], r['tag_category'])
    lines.append(f'| {label} | {r["files_using"]} | {r["usage_ratio"]:.1%} | {r["total_occurrences"]:,} |')
display(Markdown('\n'.join(lines)))

### Table 3: 高特效需求番剧 TOP 15

In [ ]:
rows = conn.execute('''
    SELECT f.anime_name, f.season, COUNT(*) AS file_count,
           ROUND(AVG(e.complexity_score),2) AS avg_cx,
           SUM(e.tag_draw) AS td, SUM(e.tag_karaoke) AS tk,
           SUM(e.tag_transform) AS tt, SUM(e.tag_move) AS tm,
           SUM(e.tag_clip) AS tc, SUM(e.total_effect_tags) AS tfx
    FROM file_stats f JOIN effect_stats e ON f.id=e.file_id
    GROUP BY f.season, f.anime_name ORDER BY avg_cx DESC LIMIT 15''').fetchall()

display(Markdown('### Table 3: 高特效需求番剧 TOP 15'))
lines = ['| 排名 | 番剧名称 | 季度 | 文件数 | 平均复杂度 | 绘图标签 | 卡拉OK | 变换 | 移动 | 裁剪 | 总特效 |',
         '|------|----------|------|--------|-----------|----------|--------|------|------|------|--------|']
for i, r in enumerate(rows, 1):
    lines.append(f'| {i} | {r["anime_name"]} | {r["season"]} | {r["file_count"]} | {r["avg_cx"]} | '
                 f'{r["td"]:,} | {r["tk"]} | {r["tt"]} | {r["tm"]} | {r["tc"]} | {r["tfx"]:,} |')
display(Markdown('\n'.join(lines)))

### Table 4: 语言类型 × 特效交叉分析

In [ ]:
rows = conn.execute('''
    SELECT f.lang_type, COUNT(*) AS file_count,
           ROUND(AVG(f.total_lines),0) AS avg_lines,
           ROUND(AVG(f.dialogue_count),0) AS avg_dl, ROUND(AVG(f.style_count),1) AS avg_st,
           ROUND(AVG(e.total_effect_tags),1) AS avg_fx, ROUND(AVG(e.complexity_score),2) AS avg_cx,
           SUM(CASE WHEN e.tag_draw>0 THEN 1 ELSE 0 END) AS df,
           SUM(CASE WHEN e.tag_karaoke>0 THEN 1 ELSE 0 END) AS kf
    FROM file_stats f JOIN effect_stats e ON f.id=e.file_id
    GROUP BY f.lang_type ORDER BY file_count DESC''').fetchall()

LANG_CN = {'chs':'简体中文','cht':'繁体中文','jpn':'纯日文','chs_jpn':'简日双语','cht_jpn':'繁日双语'}

display(Markdown('### Table 4: 语言类型 × 特效交叉分析'))
lines = ['| 语言类型 | 文件数 | 平均行数 | 平均Dialogue | 平均Style | 平均特效标签 | 平均复杂度 | 含绘图 | 含卡拉OK |',
         '|----------|--------|---------|-------------|----------|-------------|-----------|--------|----------|']
for r in rows:
    label = LANG_CN.get(r['lang_type'], r['lang_type'])
    lines.append(f'| {label} | {r["file_count"]} | {r["avg_lines"]} | {r["avg_dl"]} | '
                 f'{r["avg_st"]} | {r["avg_fx"]} | {r["avg_cx"]} | '
                 f'{r["df"]} | {r["kf"]} |')
display(Markdown('\n'.join(lines)))

### Table 5: 季度特效演变趋势

In [ ]:
rows = conn.execute('''
    SELECT season, file_count, avg_complexity, files_with_draw, files_with_karaoke,
           ROUND(CAST(files_with_draw AS REAL)/file_count*100,1) AS draw_pct,
           ROUND(CAST(files_with_karaoke AS REAL)/file_count*100,1) AS karaoke_pct,
           avg_fx_per_dialogue, fx_dialogue_ratio
    FROM season_summary ORDER BY season''').fetchall()

display(Markdown('### Table 5: 各季度特效演变趋势'))
lines = ['| 季度 | 文件数 | 平均复杂度 | 含绘图(数/%) | 含卡拉OK(数/%) | 行均特效标签 | 特效行占比 |',
         '|------|--------|-----------|-------------|---------------|-------------|------------|']
for r in rows:
    lines.append(f'| {r["season"]} | {r["file_count"]} | {r["avg_complexity"]} | '
                 f'{r["files_with_draw"]} / {r["draw_pct"]}% | '
                 f'{r["files_with_karaoke"]} / {r["karaoke_pct"]}% | '
                 f'{r["avg_fx_per_dialogue"]} | {r["fx_dialogue_ratio"]:.1%} |')
display(Markdown('\n'.join(lines)))

### Table 6: 特效复杂度分档分布

In [ ]:
TIERS = [('极简',0.0,0.0),('简单',0.1,2.0),('中等',2.1,4.0),('复杂',4.1,7.0),('极复杂',7.1,10.0)]
total = conn.execute('SELECT COUNT(*) FROM effect_stats').fetchone()[0]

display(Markdown('### Table 6: 特效复杂度分档分布'))
lines = ['| 分档 | 分数区间 | 文件数 | 占比 | 说明 |',
         '|------|---------|--------|------|------|']
for label, lo, hi in TIERS:
    if hi == 0.0:
        cnt = conn.execute('SELECT COUNT(*) FROM effect_stats WHERE complexity_score=0').fetchone()[0]
    else:
        cnt = conn.execute('''SELECT COUNT(*) FROM effect_stats
                              WHERE complexity_score>=? AND complexity_score<=?''', (lo,hi)).fetchone()[0]
    pct = cnt/total*100
    desc_map = {0.0:'纯 Dialogue，无覆盖标签',2.0:'少量颜色/位置/边框',4.0:'含 fade/move/clip',7.0:'含 karaoke/transform',10.0:'含 \\p 矢量绘图'}
    lines.append(f'| {label} | {lo}-{hi} | {cnt} | {pct:.2f}% | {desc_map[hi]} |')
lines.append(f'\n**总计: {total} 文件**')
display(Markdown('\n'.join(lines)))
conn.close()

---

## 完成

| 表格 | 维度 |
|------|------|
| Table 1 | 季度汇总 |
| Table 2 | 15 种特效标签分布 |
| Table 3 | 高特效番剧 TOP 15 |
| Table 4 | 语言类型 × 复杂度 |
| Table 5 | 季度演变趋势 |
| Table 6 | 复杂度 5 档分档 |

数据已保存至 `ass_stats.db`，可运行 `ass_charts.ipynb` 生成可视化图表。